In [13]:

from dotenv import load_dotenv, find_dotenv
import os

from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_openai import ChatOpenAI



from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.caches import InMemoryCache
from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder
)

import langchain
from pydantic import BaseModel, Field
from typing import List

from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import *
from langchain_text_splitters import CharacterTextSplitter


from langchain.tools import tool
from langchain.agents import create_agent



In [2]:
load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
@tool("search_web")
def searchWeb(input: str) -> str:
    """tool that web search for web accept one parameter named input.
    
    Args: 
        input: search for web

    Return:
        search output
    """
    print('i am in web search method')
    return {"results": [f"Job posting found on linkedin"]}

In [4]:
llm = ChatOpenAI(model="gpt-5-nano-2025-08-07", openai_api_key=OPENAI_API_KEY)

In [5]:
tools = [searchWeb]

In [ ]:
agent = create_agent(
    model=llm, 
    tools=tools
    )


In [7]:
result = agent.invoke(
        {
            "messages": HumanMessage(
                content="web search for job postings"
            )
        }
    )
print(result)

i am in web search method
{'messages': [HumanMessage(content='web search for job postings', additional_kwargs={}, response_metadata={}, id='aca4a4ef-1161-4abf-b4c2-d0082002755c'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 409, 'prompt_tokens': 149, 'total_tokens': 558, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DNauYL5MYED2BX0CbcQ1HMfU5nq9x', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d2962-4fd9-7793-9175-94589cb91a19-0', tool_calls=[{'name': 'search_web', 'args': {'input': 'current job postings'}, 'id': 'call_vN7kbXsqOJGuag86WdLbssBn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

In [10]:
class Source(BaseModel):
    """Schema for a source used by the agent"""

    url: str = Field(description="The URL of the source")


In [14]:
class AgentResponse(BaseModel):
    """Schema for agent response with answer and sources"""

    answer: str = Field(description="Thr agent's answer to the query")
    sources: List[Source] = Field(
        default_factory=list, description="List of sources used to generate the answer"
    )

In [ ]:
agent1 = create_agent(
    model=llm, 
    tools=tools, 
    response_format=AgentResponse
    )


In [ ]:
def main():
    print("Hello from langchain-course!")
    result = agent1.invoke(
        {
            "messages": HumanMessage(
                content="web search to find a job on linkedin"
            )
        }
    )
    print(result)